# From Data to Defense: Forecasting System Threats using Machine Learning!

## Description

This competition aims to predict the chances of a system getting infected by malware using Hardware properties, Software & Security configurations and regional data. The data comes from antivirus software, which tracks system settings and security events. By analyzing this data, we can build a model to identify systems at risk of malware attacks.

# Loading Necessary Libraries

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, BaggingClassifier
from lightgbm import LGBMClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
import scipy.stats as stats
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import RFECV
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
import math
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)



# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Data Exploration

In [ ]:
train_data_original= pd.read_csv("/kaggle/input/System-Threat-Forecaster/train.csv")
test_data_original = pd.read_csv("/kaggle/input/System-Threat-Forecaster/test.csv")

In [ ]:
train_data = train_data_original.copy()
test_data = test_data_original.copy()

In [ ]:
train_data.shape

In [ ]:
train_data.info()

+ We are working with int64, float64 and object datatypes.
+ Also, there are null values present in the dataset.

In [ ]:
train_data.head()

In [ ]:
train_data.describe()

In [ ]:
pd.options.display.max_rows = None
(train_data.isna().sum() / train_data.shape[0]) * 100

+ There are missing values present in the dataset.

## Visualizing Missing Values Count

In [ ]:
missing_values = train_data.isnull().sum()
missing_values = missing_values[missing_values > 0]

missing_df = pd.DataFrame({'Column': missing_values.index, 'Missing Count': missing_values.values})
missing_df = missing_df.sort_values(by='Missing Count', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x='Missing Count', y='Column', data=missing_df, palette='viridis')
plt.xlabel("Count of Missing Values")
plt.ylabel("Columns")
plt.title("Missing Values per Column")
plt.show()

+ SMode has the highest number of missing values.

In [ ]:
train_data.duplicated().sum()

+ 165 duplicate values present in the dataset. We need to remove them.

In [ ]:
train_data.drop_duplicates(inplace=True)

### Visualizing Feature Cardinality

In [ ]:
cardinality = train_data.nunique().sort_values(ascending=False)


plt.figure(figsize=(12, 6))
cardinality.plot(kind='bar', color='blue')
plt.xlabel("Features")
plt.ylabel("Unique Values Count")
plt.title("Feature Cardinality (Unique Values per Column)")
plt.xticks(rotation=90)
plt.show()


cardinality = train_data.drop(columns=["MachineID", "SystemVolumeCapacityMB"], axis=1).nunique().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
cardinality.plot(kind='bar', color='blue')
plt.xlabel("Features")
plt.ylabel("Unique Values Count")
plt.title("Feature Cardinality (Unique Values per Column)")
plt.xticks(rotation=90)
plt.show()


Observations:

+ MachineID has very high cardinality. We can drop it as it only represents unique IDs to the rows, so it is of no use.
+ CityID, OEMModelID are also showing very high cardinality with ~16,000 unique Values.
+ FirmwareID has more than 12,000 unique values.
+ Other ID columns has around 2,000 unique values and apart from ID columns, some features has around 300 unique values. We need to look at how to handle these.

In [ ]:
train_data.drop(columns=['MachineID'], axis=1, inplace=True)
test_data.drop(columns=["MachineID"], axis=1, inplace=True)

## Separating Numerical, Categorical and Binary Columns

+ To distinguish and make plotting easier, I separate the features as Numerical, Categorical, Binary columns

In [ ]:
categorical_columns = [col for col in train_data.columns if train_data[col].dtype == 'object']

binary_columns = [ col for col in train_data.columns if train_data[col].nunique() == 2 ]

numerical_columns = [ col for col in train_data.columns if train_data[col].dtype in ['int64', 'float64'] and col not in binary_columns ]

In [ ]:
column_counts = { 'category': ['Numerical', 'Categorical', 'Binary'], 'count': [len(numerical_columns), len(categorical_columns), len(binary_columns)] }

column_counts = pd.DataFrame(column_counts)

sns.barplot(x='category', y='count', data=column_counts)

plt.title('Column types count')
plt.ylabel('Counts')
plt.xlabel('Category')
plt.show()

Observations:
+ There are most number of numerical columns, followed by categorical and lastly binary

In [ ]:
categorical_columns = [col for col in train_data.columns if train_data[col].dtype == 'object']

binary_columns = [ col for col in train_data.columns if train_data[col].nunique() == 2 ]

ID_columns = [ col for col in train_data.columns if train_data[col].dtype in ['int64', 'float64'] and any(keyword in col for keyword in ["ID", "Identifier"])]

numerical_columns = [ col for col in train_data.columns if train_data[col].dtype in ['int64', 'float64'] and col not in binary_columns and col not in ID_columns ]

In [ ]:
column_counts = { 'category': ['Numerical', 'Categorical', 'Binary', 'ID'], 'count': [len(numerical_columns), len(categorical_columns), len(binary_columns), len(ID_columns)] }

column_counts = pd.DataFrame(column_counts)

sns.barplot(x='category', y='count', data=column_counts, palette='viridis')

plt.title('Column types count')
plt.ylabel('Counts')
plt.xlabel('Category')
plt.show()

Observations:
+ After separation of ID columns, we have most number of categorical columns. ID can also be considered as categorical, but its numeric in nature.

In [ ]:
sns.countplot(x='target', data=train_data, palette='viridis')
plt.title('Distribution of Target Variable')
plt.xlabel('Target')
plt.ylabel('Count')
plt.show()

Observations:
+ Target variable is balanced which is nice!

# Exploratory Data Analysis

## 1. Univariate Analysis

### Histograms for Numeric Columns:

In [ ]:
num_cols = len(numerical_columns)
num_rows = (num_cols + 2) // 3

fig, axes = plt.subplots(num_rows, 3, figsize=(15, 5 * num_rows))
axes = axes.flatten()

for idx, col in enumerate(numerical_columns):
    sns.histplot(train_data[col], ax=axes[idx], kde=True, color='red')
    axes[idx].set_title(f'Distribution of {col}')

for ax in axes[num_cols:]:
    fig.delaxes(ax)

plt.tight_layout()
plt.show()

In [ ]:
print(train_data['IsBetaUser'].value_counts())
print("-------------------------------")
print(train_data['AutoSampleSubmissionEnabled'].value_counts())
print("-------------------------------")
print(train_data['IsFlightsDisabled'].value_counts())

+ These features just have Constant values. We can drop these.

##### Observations:


1) System Security Variables:

   Variables - RealTimeProtectionState, NumAntivirusProductsEnabled, EnableLUA


  + Most systems have real-time protection enabled, and only 1-2 antivirus products configured.
  + EnableLUA is concentrated on a single value, suggesting most devices operate with default User Access Control settings.
  
 Insights: Security Configurations in this dataset are largely uniform, with minor variability.


2) Hardware Specs:

  Variables - ProcessorCoresCount, TotalPhysicalRAM, PrimaryDiskCapacity, InternalBatteryNumberOfCharges

   
  + Most systems have 4 core processors, and low-to-moderate amount of RAM, with outliers representing high performance setups.
  + Internal battery charge counts is heavily skewed to left, indicating a good number of relatively new devices.

  
  Insights: The dataset is dominated with mid-range systems with standard hardware configs and some high-performance outliers.


3) Software and System Versions:

  Variables - OSBuildNumber, OSBuildRevisionOnly, FirmwareManufacturerID, OSProductSuite, IEVersionID


  + OSBuildNumber and OSBuildRevisionOnly show a broad distribution, indicating diversity in operating system versions.
  + Internet Explorer versions are concentrated around a few values, likely representing default installations.


  Insights: Software configurations and firmware setups are diverse but tend to follow common standards with a few rare outliers.


4) System Performance:

  Variables - SystemVolumeCapacity, PrimaryDisplayDiagonalInches, PrimaryDisplayResolutionHorizontal/Vertical

  + Most systems have moderate storage capacities, with few devices exhibiting significantly higher or lower values.
  + Screen sizes and resolutions cluster around standard configurations, suggesting uniformity in display hardware.


  Insights: System performance-related variables indicate a prevalence of mid-range setups with standard display and storage configurations
Converting into numbers.

### Density Plot

+ To observe the true distribution shape without the clutter of histograms, I made Density Plots.
+ Histogram + KDE helps see frequency, while KDE-only provides a smoother density estimation.

In [ ]:
num_cols = len(numerical_columns)
num_rows = (num_cols + 2) // 3

fig, axes = plt.subplots(num_rows, 3, figsize=(15, 5 * num_rows))
axes = axes.flatten()

for idx, col in enumerate(numerical_columns):
    sns.kdeplot(train_data[col], ax=axes[idx], shade=True, color='orange')
    axes[idx].set_title(f'Density Plot of {col}')

for ax in axes[num_cols:]:
    fig.delaxes(ax)

plt.tight_layout()
plt.show()

### Histplot for ID columns

In [ ]:
num_rows = (len(ID_columns)) + 2 // 3
fig, axes = plt.subplots(num_rows, 3, figsize=(15, 5 * num_rows))
axes = axes.flatten()

for idx, col in enumerate(ID_columns):
    sns.histplot(train_data[col], ax=axes[idx], color='blue', kde=False)
    axes[idx].set_title(f'Distribution of {col}')

for idx in range(len(ID_columns), len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.show()

Observations:

+ OEMModelID, FirmwareVersionID, CityID have a large number of unique values.
+ OSUILocaleID, OSInstallLanguageID, ProcessorManufacturerID have a single category dominating the distribution.
+ A few antivirus configurations are very common, while others are rarely used.
+ Certain countries and cities dominate in the dataset, showing uneven representation across the regions.
+ GeoRegionID is relatively a uniform distribution, with few peak regions indicating concentrated device presence.
+ The dataset is regionally skewed, with a few locations contributing disproportionately to the sample.
+ FirmwareManufacturerID is dominated by a few manufacturers, with some rare configurations.

### Boxplots for Numerical Columns

In [ ]:
boxplot_cols = [ col for col in train_data.columns if train_data[col].dtype in ["int64", 'float64'] and col not in binary_columns and col not in ID_columns]

def plot_boxplots(data, boxplot_cols):
    num_cols = len(boxplot_cols)
    num_rows = (num_cols + 2) // 3

    plt.figure(figsize=(18, 5 * num_rows))

    for i, column in enumerate(boxplot_cols, 1):
        plt.subplot(num_rows, 3, i)
        sns.boxplot(data=data, x=column, palette='viridis')
        plt.title(f'Box Plot of {column}')
        plt.xlabel(column)
        plt.ylabel('Value')

    plt.tight_layout()
    plt.show()


plot_boxplots(train_data, boxplot_cols)


Observations:

+ Many features have extreme values, which can be considered as outliers but these can potentially be high-end devices of gamers, since we have a IsGamer category, Or these can be Servers and Workstations.
+ I think we can keep them, as they seem to be more of special device configurations, rather than data inconsistencices.

### Count plots for Categorical Columns:

In [ ]:
plot_categorical_columns = [ col for col in train_data.columns if train_data[col].dtype in ['object'] and col not in ['EngineVersion', 'AppVersion', 'OSBuildLab', 'NumericOSVersion', 'SignatureVersion', 'DateAS', 'DateOS']]

def categorical_plot(data, cat_columns):
    num_rows = math.ceil(len(cat_columns) / 2)

    plt.figure(figsize=(15, 5 * num_rows))

    for i, col in enumerate(cat_columns):
        plt.subplot(num_rows, 2, i + 1)
        sns.countplot(data=data, x=col, palette='plasma')
        plt.title(f'Count Plot of {col}')
        plt.xlabel(col)
        plt.ylabel('Count')
        plt.xticks(rotation=45)

    plt.tight_layout(pad=0.4, w_pad=0.5, h_pad=1.5)
    plt.show()

categorical_plot(train_data, plot_categorical_columns)

Observations :
  
+ Dataset seems to he highly skewed, with dominant categoricals across variables. E.g.: "windows8defender" for ProductName, "windows10" for PlatformType, and "x64" for Processor.

+ Several features lack diversity, such as DeviceFamily (mostly "Windows.Desktop") and FlightRing (primarily "Retail"). Similarly, categories like OSGenuineState ("IS_GENUINE") and OSArchitecture ("x64") dominate their respective distributions.

+ Some attributes show significant imbalances, such as PrimaryDiskType (high "HDD" usage over "SSD") and MDC2FormFactor (mostly "Notebook" or "Desktop").

+ Variables like OSVersion, OsPlatformSubRelease, and PowerPlatformRole show limited variety, with a few categories accounting for nearly all records while others are negligible.

+ Features such as ChassisType, OSEdition, and LicenseActivationChannel have underrepresented values.

#### Visualizing top 10 Versions by Volume

+ Due to high cardinality, I plot top 10 versions by volume

In [ ]:
top_10_engine_version = train_data['EngineVersion'].value_counts().sort_values(ascending=False)[:10]
top_10_signature_version = train_data['SignatureVersion'].value_counts().sort_values(ascending=False)[:10]
top_10_osbuildlab = train_data['OSBuildLab'].value_counts().sort_values(ascending=False)[:10]
top_10_app_version = train_data['AppVersion'].value_counts().sort_values(ascending=False)[:10]
top_10_num_osversion = train_data['NumericOSVersion'].value_counts().sort_values(ascending=False)[:10]

In [ ]:
def plot_top_10(data, column, title):
    plt.figure(figsize=(12, 6))

    top_10 = data[column].value_counts().head(10)

    sns.barplot(x=top_10.index, y=top_10.values, palette='plasma')

    plt.title(title)
    plt.xlabel(column)
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


plot_top_10(train_data, "EngineVersion", 'Top 10 Engine Version')
plot_top_10(train_data, "AppVersion", 'Top 10 App Version')
plot_top_10(train_data, "SignatureVersion", 'Top 10 Signature Version')
plot_top_10(train_data, "OSBuildLab", 'Top 10 OSBuildLab Version')
plot_top_10(train_data, "NumericOSVersion", 'Top 10 NumericOS Version')

Observations :

+ Two Engine Versions '1.1.15200.1' & '1.1.15100.1' are dominant.
+ Version '4.18.1807.18075' is the dominant and most popular App Version.
+ Signature Version has no major dominance of any particular version, looking fair.
+ 17134.1.amd64fre.rs4_release.180410-1804 is the top OSBuildLab Version.
+ 10.0.17134.228 is the top NumericOSVersion.

### Count Plots for Binary Columns

In [ ]:
binary_columns = [ col for col in train_data.columns if train_data[col].nunique() == 2 ]

def plot_binary_cols(data, binary_columns):
    num_rows = len(binary_columns) // 2 + len(binary_columns) % 2

    plt.figure(figsize=(15, 5 * num_rows))

    for i, col in enumerate(binary_columns, 1):
        plt.subplot(num_rows, 2, i)
        sns.countplot(data=data, x=col, palette='plasma')
        plt.title(f'Count Plot of {col}')
        plt.xlabel(col)
        plt.ylabel('Count')
        plt.xticks(rotation=45)

    plt.tight_layout()
    plt.show()

plot_binary_cols(train_data, binary_columns)

Observations :
+ The distributions are highly skewed
+ ProductName, HasTPM, SMode, DeviceFamily, IsVirtualDevice, IsPortableOS, IsPassiveModeEnabled are very highly skewed.
+ Overall, the distributions are skewed in nature, except target and IsSecureBootEnabled.

### Dropping redundant features based on above analysis

In [ ]:
train_data.drop(columns=['IsBetaUser', 'AutoSampleSubmissionEnabled', 'IsFlightsDisabled', 'SMode', 'HasTpm', 'IsVirtualDevice', 'IsPortableOS', "DeviceFamily", 'EnableLUA'], axis=1, inplace=True)
test_data.drop(columns=['IsBetaUser', 'AutoSampleSubmissionEnabled', 'IsFlightsDisabled', 'SMode', 'HasTpm', 'IsVirtualDevice', 'IsPortableOS', 'DeviceFamily', 'EnableLUA'], axis=1, inplace=True)

## 2. Bivariate Analysis

### Count plots grouped by Target variable for categorical columns

In [ ]:
plot_categorical_columns = [ col for col in train_data.columns if train_data[col].dtype in ['object'] and col not in ['EngineVersion', 'AppVersion', 'OSBuildLab', 'NumericOSVersion', 'SignatureVersion', 'DateAS', 'DateOS']]

def plot_categorical_by_target(df, target_col, cat_columns):
    num_rows = math.ceil(len(cat_columns) / 2)

    plt.figure(figsize=(15, 5 * num_rows))

    for i, col in enumerate(cat_columns):
        plt.subplot(num_rows, 2, i + 1)
        sns.countplot(data=df, x=col, hue=target_col, palette="seismic")
        plt.title(f'{col} grouped by target')
        plt.xticks(rotation=45)

    plt.tight_layout()
    plt.show()

plot_categorical_by_target(train_data, train_data['target'], plot_categorical_columns)

Observations:

+ For majority of the features, the target variable distribution is almost balanced.

Plotting these graphs with count on y-axis makes it unable to understand the distribution of lower frequency values. So, I will plot Rank Proportion Plot to understand the target variable distribution more.

In [ ]:
def rnp_plot(data, categorical_columns, target):
    num_plots = len(categorical_columns)
    num_rows = math.ceil(num_plots / 2)

    fig, axes = plt.subplots(nrows=num_rows, ncols=2, figsize=(12, 4 * num_rows))
    axes = axes.flatten()

    for i, column in enumerate(categorical_columns):
        category_percentages = data.groupby(column)[target].mean().sort_values(ascending=False) * 100

        bar_plot = sns.barplot(y=category_percentages.index, x=category_percentages.values, palette="viridis", ax=axes[i])
        axes[i].set_title(f'Percentage by {column}')
        axes[i].set_xlabel(f'Percentage of {target} = 1')
        axes[i].set_ylabel(column)

        for p in bar_plot.patches:
            axes[i].annotate(f'{p.get_width():.1f}%',
                             (p.get_width(), p.get_y() + p.get_height() / 2),
                             ha='left', va='center', fontsize=10, color='black')

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

rnp_plot(train_data, plot_categorical_columns, 'target')

Observations:

+ Windows 8 Defender has a higher malware hit-rate compared to mse
+ PlatformType:
>+ Windows 8 has highest hit-rate, followed by Windows 10.
>+ Windows 2016 has the lowest hit-rate, maybe people don't use that much old version.
+ Processor:
>+ x64 processors have the highest hit-rate, followed by x86.
>+ ARM64 has the lowest hit-rate.
+ OS Version:
>+ "rs4" & "windows81" are showing higher hit-rates, whereas "windows7" & "b2" are on the lower side.
+ SKU Edition:
>+ Enterprise Editions are seem to have higher hit-rates, so there seems to be more malware attacks in organizations PCs.
>+ Home is showing lower hit-rates, which are used personally by many.
+ Form Factor & Chassis Type:
>+ Desktop and Notebooks have higher malware hits whereas tablets and servers have lower malware hit-rate.
+ Primary Disk:
>+ PCs with HDD seems to have higher hit-rates compared to SSD
+ Power Platform Role:
>+ Performance Servers and Workstations show higher hit-rates

+ General Insights:
>+ Desktop and notebook devices have a higher likelihood of the target = 1.
>+ x64 architecture is dominant in terms of target = 1 occurrences.
>+ Certain OS versions, enterprise editions, and activation channels correlate with higher target values.
>+ HDDs are more likely associated with target = 1 than SSDs.
>+ Enterprise and corporate licensing models show higher target percentages.

In [ ]:
def top_10_columns_plot(top_values, target, title, x, y):

    filtered_data = train_data[train_data[x].isin(top_values.index)]

    count_data = filtered_data.groupby([x, target]).size().reset_index(name='count')

    plt.figure(figsize=(12, 6))
    sns.barplot(data=count_data, x=x, y='count', hue=target, palette='seismic')

    plt.title(title)
    plt.xlabel(x)
    plt.ylabel('Frequency (Count)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

top_10_columns_plot(top_10_engine_version, 'target', "Top 10 Engine Version", 'EngineVersion', 'Count')
top_10_columns_plot(top_10_signature_version, 'target', "Top 10 Signature Version", "SignatureVersion", "Count")
top_10_columns_plot(top_10_app_version, 'target', "Top 10 App Version", "AppVersion", "Count")
top_10_columns_plot(top_10_osbuildlab, 'target', "Top 10 OSBuildLab Version", "OSBuildLab", "Count")
top_10_columns_plot(top_10_num_osversion, 'target', "Top 10 Numeric OS Version", "NumericOSVersion", "Count")

Observations:

+ No significant variation in target variable for EngineVersion.
+ In SignatureVersion, 1.273.420.0 is showing more 1's and 1.273.1140.0 showing more instance of 0's, suggesting some correlation with target due to slight skewness.
+ No significant variation in target variable for AppVersion.
+ No significant variation in target variable for OSBuildLab
+ In NumericOSVersion, '10.0.17134.165' & '10.0.17134.191' show higher proportion of target '1', suggests these two versions may correlate with target '1'.

In [ ]:
def rnp_version_plot(data, version_columns, target):
    num_plots = len(version_columns)
    num_rows = math.ceil(num_plots / 2)

    fig, axes = plt.subplots(nrows=num_rows, ncols=2, figsize=(14, 4 * num_rows))
    axes = axes.flatten()

    for i, column in enumerate(version_columns):
        top_10_categories = data[column].value_counts().nlargest(10).index
        filtered_data = data[data[column].isin(top_10_categories)]

        category_percentages = (
            filtered_data.groupby(column)[target].mean().sort_values(ascending=False) * 100
        )

        bar_plot = sns.barplot(y=category_percentages.index, x=category_percentages.values, palette="viridis", ax=axes[i])
        axes[i].set_title(f'Percentage of {target} = 1 by {column}')
        axes[i].set_xlabel(f'Percentage of {target} = 1')
        axes[i].set_ylabel(column)

        for p in bar_plot.patches:
            axes[i].annotate(f'{p.get_width():.1f}%',
                             (p.get_width(), p.get_y() + p.get_height() / 2),
                             ha='left', va='center', fontsize=10, color='black')

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

version_columns = ['EngineVersion', "SignatureVersion", "AppVersion", "NumericOSVersion", "OSBuildLab"]

rnp_version_plot(train_data, version_columns, 'target')

Observations:

+ No major incline towards malware hit. All are below 60%.
+ EngineVersion - The highest percentage is observed for version 1.1.15300.5, suggesting that it has a strong association with the malware attack.
+ Signature versions 1.273.1749.0 and 1.273.1420.0 show the highest percentages for malware hit-rate.
+ The highest impact is seen for app version 4.18.1807.18075, and few versions like 4.13.17134.1 and 4.10.209.0 also have a significant percentage.
+ OS version 10.0.17134.165 has the highest probability of malware hit.
+ The OS build 17134.1.amd64fre.rs4_release.180410-1804 has the highest .


### Count plots for Binary Columns grouped by Target variable

In [ ]:
binary_columns = [ col for col in train_data.columns if train_data[col].nunique() == 2 ]
def plot_binary_countplots(data, binary_columns):
    num_rows = len(binary_columns) // 2

    plt.figure(figsize=(15, 5 * num_rows))

    for i, col in enumerate(binary_columns[:-1], 1):
        plt.subplot(num_rows, 2, i)
        sns.countplot(data=data, x=col, hue='target', palette='seismic')
        plt.title(f'Count Plot of {col} grouped by target')
        plt.xlabel(col)
        plt.ylabel('Count')
        plt.xticks(rotation=45)

    plt.tight_layout()
    plt.show()

plot_binary_countplots(train_data, binary_columns)

Observations:

+ Many features have strong class imbalances.
+ Most of the features have a balanced target distribution, means that these features may not strongly differentiate between the targets on their own.

In [ ]:
def rnp_binary_plot(data, binary_columns, target):
    num_plots = len(binary_columns)
    num_rows = math.ceil(num_plots / 2)

    fig, axes = plt.subplots(nrows=num_rows, ncols=2, figsize=(12, 4 * num_rows))
    axes = axes.flatten()

    for i, column in enumerate(binary_columns):
        category_percentages = data.groupby(column)[target].mean() * 100

        bar_plot = sns.barplot(
            y=category_percentages.index.astype(str),
            x=category_percentages.values,
            palette="viridis",
            ax=axes[i]
        )

        axes[i].set_title(f'Percentage of {target} = 1 by {column}')
        axes[i].set_xlabel(f'Percentage of {target} = 1')
        axes[i].set_ylabel(column)

        for p in bar_plot.patches:
            axes[i].annotate(f'{p.get_width():.1f}%',
                             (p.get_width(), p.get_y() + p.get_height() / 2),
                             ha='left', va='center', fontsize=10, color='black')

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

binary_columns = [col for col in train_data.columns if train_data[col].nunique() == 2 and col != 'target']

rnp_binary_plot(train_data, binary_columns, 'target')

Observations:

+ Windows 8 showing more malware hit-rate.
+ Devices with PassiveModeEnabled are showing lower malware hit-rate compared to those with disabled.
+ Protected Systems are showing more malware attacks compared to unprotected.
+ Systems with Firewall Enabled are showing slightly more malware attacks.
+ More attacks are seen in non-touch & non-pencapable devices, so mostly desktops.
+ Gamer's devices are showing higher malware hit-rates.

### Heat Maps

### Pearson Correlation

In [ ]:
numerical_columns = [ col for col in train_data.columns if train_data[col].dtype in ['int64', 'float64'] ]

numerical_columns_df = train_data[numerical_columns]

def plot_heatmap(data, corr_matrix=None):

    if corr_matrix is None:
        corr_matrix = data.corr()

    plt.figure(figsize=(20, 20))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
    plt.title('Correlation Heatmap')
    plt.tight_layout()
    plt.show()

plot_heatmap(numerical_columns_df)

Observations:

+ OSBuildNumber and OSBuildLabOnly show very high correlations of 0.95.
+ OSUILocaleID and OSInstallLanguageID show very high correlations of 0.99.
+ SystemVolumeCapacity & PrimaryDiskCapacity also show a higher correlation, but they represent unique information.
+ PrimaryDisplayResolutionHorizontal & PrimaryDisplayResolutionVertical also show very high correlation of 0.90, but they represent unique information.

Potential Redundant Features from Pearson Correlation: OSBuildNumber, OSBuildLabOnly, OSUILocaleID, OSInstallLanguageID

### Cramer's V for Categorical Correlations

In [ ]:
categorical_cols = [col for col in train_data.columns if train_data[col].dtype == 'object']

def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()

    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    r_corr = r - ((r-1)**2)/(n-1)
    k_corr = k - ((k-1)**2)/(n-1)
    return np.sqrt(phi2corr / min((k_corr-1), (r_corr-1)))

categorical_corr_matrix = pd.DataFrame(index=categorical_cols, columns=categorical_cols)

for i, col1 in enumerate(categorical_cols):
    for j, col2 in enumerate(categorical_cols):
        if j <= i:
            value = cramers_v(train_data[col1], train_data[col2])
            categorical_corr_matrix.loc[col1, col2] = value

categorical_corr_matrix = categorical_corr_matrix.astype(float)

plt.figure(figsize=(15, 15))
sns.heatmap(categorical_corr_matrix, cmap='coolwarm', annot=True, fmt=".2f", linewidths=0.5)
plt.title("Cramér's V Heatmap for Categorical Features")
plt.show()

Observations:

+ PlatformType & ProductName show very high correlation of 0.99
+ SignatureVersion & EngineVersion show high correlation of 0.91
+ OSVersion, OsPlatformSubRelease, OSBuildLab are highly correlated with ProductName
+ OSBuildLab is perfectly correlated with Processor & OsPlatformSubRelease.
+ OSArchitecture perfectly correlated with Processor & OSBuildLab.
+ OSBranch show very high correlation of 0.96 with OSBuildLab & NumericOSVersion.
+ OSEdition, OSSkuFriendlyName, SKUEditionName are highly inter correlated.
+ DateAS & DateOS perfectly correlated with SignatureVersion & NumericOSVersion respectively.

Potential Redundant Features: OSBuildLab, OSSkuFriendlyName, SKUEditionName, OSEdition, Processor, OSArchitecture, PlatformType.


### Spearman's Rank Correlation

In [ ]:
spearman_corr = train_data.corr(method='spearman', numeric_only=True)

plt.figure(figsize=(25, 25))
sns.heatmap(spearman_corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Spearman's Rank Correlation Heatmap")
plt.show()

Observations:

+ OSBuildNumber & OSBuildNumberOnly show very high correlation again.
+ IEVersionID & OSBuildNumberOnly show some high correlation.
+ OSUILocaleID & OSInstallLanguageID again show very high correlation.

Potential Redundant Features: OSBuildNumberOnly, OSInstallLanguageID, OSUILocaleID

#### Dropping redundant features based on above analysis

In [ ]:
train_data.drop(["OSBuildLab", "OSBuildNumberOnly", "SKUEditionName", "OSSkuFriendlyName", "OSInstallLanguageID", "Processor", "OSVersion"], axis=1, inplace=True)
test_data.drop(["OSBuildLab", "OSBuildNumberOnly", "SKUEditionName", "OSSkuFriendlyName", "OSInstallLanguageID", "Processor", "OSVersion"], axis=1, inplace=True)

## EDA Summary:

The dataset primarily consists of consumer-grade devices with standard security configurations, as indicated by minimal variations in EnableLUA settings and the presence of default security measures. There is a noticeable geographic skew, with a small number of countries and cities dominating the data, which could impact global generalizability. Most devices have mid-range hardware specifications, such as quad-core processors and moderate RAM, though some high-end outliers suggest the presence of specialized systems. Software diversity is limited, with a few OS versions being predominant, and highly skewed categorical variables like ProductName and OSArchitecture indicating a lack of variation. Several features, such as OSBuildNumber, Processor, and version-based attributes, show high correlations, which could lead to redundancy in analysis. Additionally, many categorical and binary variables are imbalanced, potentially reducing their predictive significance. However, the target variable appears relatively balanced, minimizing concerns about class distribution issues.



# Feature Splitting/Transformation

In [ ]:
train_data[['EngineVersion_Major', 'EngineVersion_Minor', 'EngineVersion_Build', 'EngineVersion_Revision']] = train_data['EngineVersion'].str.split('.', expand=True).astype(float)

train_data[['AppVersion_Major', 'AppVersion_Minor', 'AppVersion_Build', 'AppVersion_Revision']] = train_data['AppVersion'].str.split('.', expand=True).astype(float)

train_data[['SignatureVersion_Major', 'SignatureVersion_Minor', 'SignatureVersion_Build', 'SignatureVersion_Revision']] = train_data['SignatureVersion'].str.split('.', expand=True).astype(float)

train_data[['NumericOS_Major', 'NumericOS_Minor', 'NumericOS_Build', 'NumericOS_Revision']] = train_data['NumericOSVersion'].str.split('.', expand=True).astype(float)

train_data['Malware_year'] = pd.to_datetime(train_data['DateAS']).dt.year
train_data['Malware_month'] = pd.to_datetime(train_data['DateAS']).dt.month
train_data['Malware_day'] = pd.to_datetime(train_data['DateAS']).dt.day
train_data['Malware_hour'] = pd.to_datetime(train_data['DateAS']).dt.hour
train_data['Malware_minute'] = pd.to_datetime(train_data['DateAS']).dt.minute

train_data['OS_year'] = pd.to_datetime(train_data['DateOS']).dt.year
train_data['OS_month'] = pd.to_datetime(train_data['DateOS']).dt.month
train_data['OS_day'] = pd.to_datetime(train_data['DateOS']).dt.weekday



test_data[['EngineVersion_Major', 'EngineVersion_Minor', 'EngineVersion_Build', 'EngineVersion_Revision']] = test_data['EngineVersion'].str.split('.', expand=True).astype(float)

test_data[['AppVersion_Major', 'AppVersion_Minor', 'AppVersion_Build', 'AppVersion_Revision']] = test_data['AppVersion'].str.split('.', expand=True).astype(float)

test_data[['SignatureVersion_Major', 'SignatureVersion_Minor', 'SignatureVersion_Build', 'SignatureVersion_Revision']] = test_data['SignatureVersion'].str.split('.', expand=True).astype(float)

test_data[['NumericOS_Major', 'NumericOS_Minor', 'NumericOS_Build', 'NumericOS_Revision']] = test_data['NumericOSVersion'].str.split('.', expand=True).astype(float)

test_data['Malware_year'] = pd.to_datetime(test_data['DateAS']).dt.year
test_data['Malware_month'] = pd.to_datetime(test_data['DateAS']).dt.month
test_data['Malware_day'] = pd.to_datetime(test_data['DateAS']).dt.day
test_data['Malware_hour'] = pd.to_datetime(test_data['DateAS']).dt.hour
test_data['Malware_minute'] = pd.to_datetime(test_data['DateAS']).dt.minute

test_data['OS_year'] = pd.to_datetime(test_data['DateOS']).dt.year
test_data['OS_month'] = pd.to_datetime(test_data['DateOS']).dt.month
test_data['OS_day'] = pd.to_datetime(test_data['DateOS']).dt.weekday

### Dropping old columns before feature splitting

In [ ]:
train_data.drop(columns=['EngineVersion', 'AppVersion', 'SignatureVersion', 'NumericOSVersion', 'DateAS', 'DateOS'], axis=1, inplace=True)

test_data.drop(columns=['EngineVersion', 'AppVersion', 'SignatureVersion', 'NumericOSVersion', 'DateAS', 'DateOS'], axis=1, inplace=True)

## Feature Binning (Grouping features to reduce cardinality)

In [ ]:
train_data.loc[train_data['MDC2FormFactor'].isin(["Desktop", "PCOther", 'AllInOne']), 'MDC2FormFactor_Grouped'] = 'Desktop'
train_data.loc[train_data['MDC2FormFactor'].isin(["Notebook", "Convertible", "Detachable"]), 'MDC2FormFactor_Grouped'] = 'Notebook'
train_data.loc[train_data['MDC2FormFactor'].isin(["LargeTablet", "SmallTablet"]), 'MDC2FormFactor_Grouped'] = 'Tablet'
train_data.loc[train_data['MDC2FormFactor'].isin(["SmallServer", "MediumServer", "LargeServer"]), 'MDC2FormFactor_Grouped'] = 'Server'


train_data.loc[train_data['PrimaryDiskType'].isin(["HDD"]), 'PrimaryDiskType_Grouped'] = 'HDD'
train_data.loc[train_data['PrimaryDiskType'].isin(["SSD"]), 'PrimaryDiskType_Grouped'] = 'SSD'
train_data.loc[~train_data['PrimaryDiskType'].isin(["HDD", 'SSD']), 'PrimaryDiskType_Grouped'] = 'Others'


train_data.loc[train_data['ChassisType'].isin(["Desktop", 'Tower', 'MiniTower', 'LowProfileDesktop', 'MiniPC', 'AllinOne']), 'ChassisType_Grouped'] = 'Desktop'
train_data.loc[train_data['ChassisType'].isin(["Notebook", "Portable", "Laptop"]), 'ChassisType_Grouped'] = 'Notebook'
train_data.loc[train_data['ChassisType'].isin(["Tablet", 'Convertible', 'Detachable', 'HandHeld']), 'ChassisType_Grouped'] = 'Tablet'
train_data.loc[~train_data['ChassisType'].isin(["Desktop", "Laptop", "Portable", 'AllinOne', 'Convertible', 'MiniTower', "Detachable", "LowProfileDesktop", 'HandHeld', 'MiniPC', 'Notebook', 'Tablet', 'Tower']), 'ChassisType_Grouped'] = 'Others'


train_data.loc[train_data['PowerPlatformRole'].isin(["Desktop"]), 'PowerPlatformRole_Grouped'] = 'Desktop'
train_data.loc[train_data['PowerPlatformRole'].isin(["Mobile", 'Slate']), 'PowerPlatformRole_Grouped'] = 'Portable'
train_data.loc[train_data['PowerPlatformRole'].isin(["SOHOServer", "EnterpriseServer", "PerformanceServer"]), 'PowerPlatformRole_Grouped'] = 'Server'
train_data.loc[~train_data['PowerPlatformRole'].isin(["Desktop", 'Mobile', 'Slate', 'SOHOServer', 'EnterpriseServer', 'PerformanceServer']), 'PowerPlatformRole_Grouped'] = 'Others'


train_data.loc[train_data['OSBranch'].isin(['rs1_release', 'rs2_release', 'rs3_release', 'rs3_release_svc_escrow', 'rs3_release_svc_escrow_im', 'rs4_release', 'rs5_release', 'rs_prerelease_flt', 'rs_prerelease']), 'OSBranch_Grouped'] = 'rs_release'
train_data.loc[train_data['OSBranch'].isin(['th1_st1', 'th1', 'th2_release', 'th2_release_sec']), 'OSBranch_Grouped'] = 'th_release'


train_data.loc[train_data['OSEdition'].isin(['Core', 'CoreSingleLanguage', 'CoreCountrySpecific', 'CoreN']), 'OSEdition_Grouped'] = 'Core'
train_data.loc[train_data['OSEdition'].isin(['Professional', 'ProfessionalN' 'ProfessionalEducation', 'Education', 'EducationN','ProfessionalEducationN' 'ProfessionalWorkstation', 'ProfessionalSingleLanguage', 'ProfessionalCountrySpecific']), 'OSEdition_Grouped'] = 'Professional'
train_data.loc[train_data['OSEdition'].isin(['Enterprise', 'EnterpriseN', 'EnterpriseS', 'EnterpriseSN']), 'OSEdition_Grouped'] = 'Enterprise'
train_data.loc[~train_data['OSEdition'].isin(['Core', 'CoreSingleLanguage', 'CoreCountrySpecific', 'CoreN', 'Professional', 'ProfessionalN' 'ProfessionalEducation', 'Education', 'EducationN','ProfessionalEducationN' 'ProfessionalWorkstation', 'ProfessionalSingleLanguage', 'ProfessionalCountrySpecific', 'Enterprise', 'EnterpriseN', 'EnterpriseS', 'EnterpriseSN']), 'OSEdition_Grouped'] = 'Others'


train_data.loc[train_data['OSInstallType'].isin(["UUPUGrade", "Update", "Upgrade"]), 'OSInstallType_Grouped'] = 'Upgrade'
train_data.loc[train_data['OSInstallType'].isin(['Reset', 'Refresh', 'CleanPCRefresh', 'Clean', 'IBSClean']), 'OSInstallType_Grouped'] = 'Clean'
train_data.loc[~train_data['OSInstallType'].isin(["UUPUGrade", "Update", "Upgrade", 'Reset', 'Refresh', 'CleanPCRefresh', 'Clean', 'IBSClean']), 'OSInstallType_Grouped'] = 'Others'


train_data.loc[train_data['AutoUpdateOptionsName'].isin(["FullAuto", "AutoInstallAndRebootAtMaintenanceTime"]), 'AutoUpdateOptionsName_Grouped'] = 'Auto'
train_data.loc[train_data['AutoUpdateOptionsName'].isin(["Notify", "DownloadNotify", "Off"]), 'AutoUpdateOptionsName_Grouped'] = 'Manual'
train_data.loc[train_data['AutoUpdateOptionsName'].isin(["Off"]), 'AutoUpdateOptionsName_Grouped'] = 'Off'
train_data.loc[~train_data['AutoUpdateOptionsName'].isin(["FullAuto", "AutoInstallAndRebootAtMaintenanceTime", 'Notify', 'DownloadNotify', 'Off']), 'AutoUpdateOptionsName_Grouped'] = 'Unknown'


train_data.loc[train_data['LicenseActivationChannel'].isin(['Retail', 'Retail:TB:Eval']), 'LicenseActivationChannel_Grouped'] = 'Retail'
train_data.loc[train_data['LicenseActivationChannel'].isin(['Volume:GVLK', 'Volume:MAK']), 'LicenseActivationChannel_Grouped'] = 'Volume'
train_data.loc[~train_data['LicenseActivationChannel'].isin(['Retail', 'Retail:TB:Eval', 'Volume:GVLK', 'Volume:MAK']), 'LicenseActivationChannel_Grouped'] = 'OEM'


train_data.loc[train_data['FlightRing'].isin(['Retail']), 'FlightRing_Grouped'] = 'Retail'
train_data.loc[train_data['FlightRing'].isin(['WIS', 'RP', 'WIF']), 'FlightRing_Grouped'] = 'Insider'
train_data.loc[train_data['FlightRing'].isin(['Disabled']), 'FlightRing_Grouped'] = 'Disabled'
train_data.loc[~train_data['FlightRing'].isin(['Retail', 'WIS', 'RP', 'WIF', 'Disabled']), 'FlightRing_Grouped'] = 'Unknown'




test_data.loc[test_data['MDC2FormFactor'].isin(["Desktop", "PCOther", 'AllInOne']), 'MDC2FormFactor_Grouped'] = 'Desktop'
test_data.loc[test_data['MDC2FormFactor'].isin(["Notebook", "Convertible", "Detachable"]), 'MDC2FormFactor_Grouped'] = 'Notebook'
test_data.loc[test_data['MDC2FormFactor'].isin(["LargeTablet", "SmallTablet"]), 'MDC2FormFactor_Grouped'] = 'Tablet'
test_data.loc[test_data['MDC2FormFactor'].isin(["SmallServer", "MediumServer", "LargeServer"]), 'MDC2FormFactor_Grouped'] = 'Server'


test_data.loc[test_data['PrimaryDiskType'].isin(["HDD"]), 'PrimaryDiskType_Grouped'] = 'HDD'
test_data.loc[test_data['PrimaryDiskType'].isin(["SSD"]), 'PrimaryDiskType_Grouped'] = 'SSD'
test_data.loc[~test_data['PrimaryDiskType'].isin(["HDD", 'SSD']), 'PrimaryDiskType_Grouped'] = 'Others'


test_data.loc[test_data['ChassisType'].isin(["Desktop", 'Tower', 'MiniTower', 'LowProfileDesktop', 'MiniPC', 'AllinOne']), 'ChassisType_Grouped'] = 'Desktop'
test_data.loc[test_data['ChassisType'].isin(["Notebook", "Portable", "Laptop"]), 'ChassisType_Grouped'] = 'Notebook'
test_data.loc[test_data['ChassisType'].isin(["Tablet", 'Convertible', 'Detachable', 'HandHeld']), 'ChassisType_Grouped'] = 'Tablet'
test_data.loc[~test_data['ChassisType'].isin(["Desktop", "Laptop", "Portable", 'AllinOne', 'Convertible', 'MiniTower', "Detachable", "LowProfileDesktop", 'HandHeld', 'MiniPC', 'Notebook', 'Tablet', 'Tower']), 'ChassisType_Grouped'] = 'Others'


test_data.loc[test_data['PowerPlatformRole'].isin(["Desktop"]), 'PowerPlatformRole_Grouped'] = 'Desktop'
test_data.loc[test_data['PowerPlatformRole'].isin(["Mobile", 'Slate']), 'PowerPlatformRole_Grouped'] = 'Portable'
test_data.loc[test_data['PowerPlatformRole'].isin(["SOHOServer", "EnterpriseServer", "PerformanceServer"]), 'PowerPlatformRole_Grouped'] = 'Server'
test_data.loc[~test_data['PowerPlatformRole'].isin(["Desktop", 'Mobile', 'Slate', 'SOHOServer', 'EnterpriseServer', 'PerformanceServer']), 'PowerPlatformRole_Grouped'] = 'Others'


test_data.loc[test_data['OSBranch'].isin(['rs1_release', 'rs2_release', 'rs3_release', 'rs3_release_svc_escrow', 'rs3_release_svc_escrow_im', 'rs4_release', 'rs5_release', 'rs_prerelease_flt', 'rs_prerelease']), 'OSBranch_Grouped'] = 'rs_release'
test_data.loc[test_data['OSBranch'].isin(['th1_st1', 'th1', 'th2_release', 'th2_release_sec']), 'OSBranch_Grouped'] = 'th_release'


test_data.loc[test_data['OSEdition'].isin(['Core', 'CoreSingleLanguage', 'CoreCountrySpecific', 'CoreN']), 'OSEdition_Grouped'] = 'Core'
test_data.loc[test_data['OSEdition'].isin(['Professional', 'ProfessionalN' 'ProfessionalEducation', 'Education', 'EducationN','ProfessionalEducationN' 'ProfessionalWorkstation', 'ProfessionalSingleLanguage', 'ProfessionalCountrySpecific']), 'OSEdition_Grouped'] = 'Professional'
test_data.loc[test_data['OSEdition'].isin(['Enterprise', 'EnterpriseN', 'EnterpriseS', 'EnterpriseSN']), 'OSEdition_Grouped'] = 'Enterprise'
test_data.loc[~test_data['OSEdition'].isin(['Core', 'CoreSingleLanguage', 'CoreCountrySpecific', 'CoreN', 'Professional', 'ProfessionalN' 'ProfessionalEducation', 'Education', 'EducationN','ProfessionalEducationN' 'ProfessionalWorkstation', 'ProfessionalSingleLanguage', 'ProfessionalCountrySpecific', 'Enterprise', 'EnterpriseN', 'EnterpriseS', 'EnterpriseSN']), 'OSEdition_Grouped'] = 'Others'


test_data.loc[test_data['OSInstallType'].isin(["UUPUGrade", "Update", "Upgrade"]), 'OSInstallType_Grouped'] = 'Upgrade'
test_data.loc[test_data['OSInstallType'].isin(['Reset', 'Refresh', 'CleanPCRefresh', 'Clean', 'IBSClean']), 'OSInstallType_Grouped'] = 'Clean'
test_data.loc[~test_data['OSInstallType'].isin(["UUPUGrade", "Update", "Upgrade", 'Reset', 'Refresh', 'CleanPCRefresh', 'Clean', 'IBSClean']), 'OSInstallType_Grouped'] = 'Others'


test_data.loc[test_data['AutoUpdateOptionsName'].isin(["FullAuto", "AutoInstallAndRebootAtMaintenanceTime"]), 'AutoUpdateOptionsName_Grouped'] = 'Auto'
test_data.loc[test_data['AutoUpdateOptionsName'].isin(["Notify", "DownloadNotify", "Off"]), 'AutoUpdateOptionsName_Grouped'] = 'Manual'
test_data.loc[test_data['AutoUpdateOptionsName'].isin(["Off"]), 'AutoUpdateOptionsName_Grouped'] = 'Off'
test_data.loc[~test_data['AutoUpdateOptionsName'].isin(["FullAuto", "AutoInstallAndRebootAtMaintenanceTime", 'Notify', 'DownloadNotify', 'Off']), 'AutoUpdateOptionsName_Grouped'] = 'Unknown'


test_data.loc[test_data['LicenseActivationChannel'].isin(['Retail', 'Retail:TB:Eval']), 'LicenseActivationChannel_Grouped'] = 'Retail'
test_data.loc[test_data['LicenseActivationChannel'].isin(['Volume:GVLK', 'Volume:MAK']), 'LicenseActivationChannel_Grouped'] = 'Volume'
test_data.loc[~test_data['LicenseActivationChannel'].isin(['Retail', 'Retail:TB:Eval', 'Volume:GVLK', 'Volume:MAK']), 'LicenseActivationChannel_Grouped'] = 'OEM'


test_data.loc[test_data['FlightRing'].isin(['Retail']), 'FlightRing_Grouped'] = 'Retail'
test_data.loc[test_data['FlightRing'].isin(['WIS', 'RP', 'WIF']), 'FlightRing_Grouped'] = 'Insider'
test_data.loc[test_data['FlightRing'].isin(['Disabled']), 'FlightRing_Grouped'] = 'Disabled'
test_data.loc[~test_data['FlightRing'].isin(['Retail', 'WIS', 'RP', 'WIF', 'Disabled']), 'FlightRing_Grouped'] = 'Unknown'



### Dropping old features before Feature Binning.

In [ ]:
train_data.drop(columns=['MDC2FormFactor', 'PrimaryDiskType', 'ChassisType', 'PowerPlatformRole', 'OSBranch', 'OSEdition', 'OSInstallType', 'AutoUpdateOptionsName', 'LicenseActivationChannel', 'FlightRing'], axis=1, inplace=True)

test_data.drop(columns=['MDC2FormFactor', 'PrimaryDiskType', 'ChassisType', 'PowerPlatformRole', 'OSBranch', 'OSEdition', 'OSInstallType', 'AutoUpdateOptionsName', 'LicenseActivationChannel', 'FlightRing'], axis=1, inplace=True)

# Feature Engineering

In [ ]:
train_data["Days_since_OS_Installation"] = (pd.to_datetime(train_data['Malware_day']) - pd.to_datetime(train_data['OS_day'])).dt.days

train_data["Ram_per_core"] = train_data["TotalPhysicalRAMMB"] / train_data["ProcessorCoreCount"]

train_data["Aspect_Ratio"] = train_data["PrimaryDisplayResolutionHorizontal"] / train_data["PrimaryDisplayResolutionVertical"]

train_data["Primary_Disk_Allocated"] = train_data["PrimaryDiskCapacityMB"] / train_data["SystemVolumeCapacityMB"]

train_data["Pixel_Density"] = (train_data["PrimaryDisplayResolutionHorizontal"] * train_data["PrimaryDisplayResolutionVertical"]) / train_data["PrimaryDisplayDiagonalInches"]

train_data["Free_Disk_Space"] = (train_data["SystemVolumeCapacityMB"] - train_data["PrimaryDiskCapacityMB"]) / train_data["PrimaryDiskCapacityMB"]





test_data["Days_since_OS_Installation"] = (pd.to_datetime(test_data['Malware_day']) - pd.to_datetime(test_data['OS_day'])).dt.days

test_data["Ram_per_core"] = test_data["TotalPhysicalRAMMB"] / test_data["ProcessorCoreCount"]

test_data["Aspect_Ratio"] = test_data["PrimaryDisplayResolutionHorizontal"] / test_data["PrimaryDisplayResolutionVertical"]

test_data["Primary_Disk_Allocated"] = test_data["PrimaryDiskCapacityMB"] / test_data["SystemVolumeCapacityMB"]

test_data["Pixel_Density"] = (test_data["PrimaryDisplayResolutionHorizontal"] * test_data["PrimaryDisplayResolutionVertical"]) / test_data["PrimaryDisplayDiagonalInches"]

test_data["Free_Disk_Space"] = (test_data["SystemVolumeCapacityMB"] - test_data["PrimaryDiskCapacityMB"]) / test_data["PrimaryDiskCapacityMB"]

## HeatMap after Feature Engineering

### Pearson Correlation

In [ ]:
categorical_columns = [col for col in train_data.columns if train_data[col].dtype == 'object']

binary_columns = [ col for col in train_data.columns if train_data[col].nunique() == 2 ]

numerical_columns = [ col for col in train_data.columns if train_data[col].dtype in ['int64', 'float64'] and col not in binary_columns ]

def plot_heatmap(data, corr_matrix=None):

    if corr_matrix is None:
        corr_matrix = data.corr()

    plt.figure(figsize=(23, 23))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
    plt.title('Correlation Heatmap')
    plt.tight_layout()
    plt.show()

corr_matrix = train_data[numerical_columns].corr()

plot_heatmap(train_data, corr_matrix)

Observations:
+ Many Constant columns got introduced. We need to drop them.
+ SignatureVersion_Minor & EngineVersion_Build show perfect correlation of 1.00
+ NumericOS_Build & OSBuildNumber show high correlation of 0.95.
+ NumericOS_Revision & OSBuildRevisionOnly show show perfect correlation of 1.00

Potential Redundant Features: SignatureVersion_Minor, NumericOS_Build, NumericOS_Revision, SignatureVersion_Major, SignatureVersion_Revision, NumericOS_Major, NumericOS_Minor, EngineVersion_Major, EngineVersion_Minor, AppVersion_Major

### Spearman's Correlation

In [ ]:
spearman_corr = train_data.corr(method='spearman', numeric_only=True)

plt.figure(figsize=(40, 40))
sns.heatmap(spearman_corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Spearman's Rank Correlation Heatmap")
plt.show()


### Cramer's V Heatmap for Categorical Correlations

In [ ]:
categorical_cols = [col for col in train_data.columns if train_data[col].dtype == 'object']

def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    r_corr = r - ((r-1)**2)/(n-1)
    k_corr = k - ((k-1)**2)/(n-1)
    return np.sqrt(phi2corr / min((k_corr-1), (r_corr-1)))

categorical_corr_matrix = pd.DataFrame(index=categorical_cols, columns=categorical_cols)

for col1 in categorical_cols:
    for col2 in categorical_cols:
        categorical_corr_matrix.loc[col1, col2] = cramers_v(train_data[col1], train_data[col2])

categorical_corr_matrix = categorical_corr_matrix.astype(float)

plt.figure(figsize=(15, 15))
sns.heatmap(categorical_corr_matrix, cmap='coolwarm', annot=True, fmt=".2f", linewidths=0.5)
plt.title("Cramér's V Heatmap for Categorical Features")
plt.show()

Observations:

+ PlatformType, OSPlatformSubRelease both are very highly correlated with ProductName.
+ OsPlatformSubRelease very highly correlated with OSBranch_Grouped.

Potential Redundant Features: OSPlatformSubRelease

#### Dropping Redundant Features based on above analysis

In [ ]:
train_data.drop(["SignatureVersion_Minor", "SignatureVersion_Major", "SignatureVersion_Revision", "AppVersion_Major", "EngineVersion_Major", "EngineVersion_Minor", "NumericOS_Major", "NumericOS_Minor", "NumericOS_Revision", "NumericOS_Build", "ProductName", "OsPlatformSubRelease", "OSBuildRevisionOnly"], axis=1, inplace=True)

test_data.drop(["SignatureVersion_Minor", "SignatureVersion_Major", "SignatureVersion_Revision", "AppVersion_Major", "EngineVersion_Major", "EngineVersion_Minor", "NumericOS_Major", "NumericOS_Minor", "NumericOS_Revision", "NumericOS_Build", "ProductName", "OsPlatformSubRelease", "OSBuildRevisionOnly"], axis=1, inplace=True)

# Data Preprocessing

In [ ]:
X = train_data.drop(columns=["target"], axis=1)
y = train_data["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
categorical_columns= [col for col in train_data.columns if train_data[col].dtype == 'object']

binary_columns = [ col for col in train_data.columns if train_data[col].nunique() == 2 and col != 'target' ]

ID_columns = [ col for col in train_data.columns if train_data[col].dtype in ['int64', 'float64'] and any(keyword in col for keyword in ["ID", "Identifier"])]

numerical_columns = [ col for col in train_data.columns if train_data[col].dtype in ['int64', 'float64'] and col not in binary_columns and col not in ID_columns and col != 'target' ]


def cast_to_float(X):
    return X.astype(float)


numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("mms", MinMaxScaler())
])

binary_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ord", OrdinalEncoder())
])

id_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])


preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_columns),
        ("bin", binary_transformer, binary_columns),
        ("id", id_transformer, ID_columns),
        ("cat", categorical_transformer, categorical_columns)
    ]
)

X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

test_data_preprocessed = preprocessor.transform(test_data)

X_train_preprocessed_df = pd.DataFrame(X_train_preprocessed, columns=numerical_columns + binary_columns + ID_columns + list(preprocessor.named_transformers_['cat']['encoder'].get_feature_names_out(categorical_columns)))

X_test_preprocessed_df = pd.DataFrame(X_test_preprocessed, columns=numerical_columns + binary_columns + ID_columns + list(preprocessor.named_transformers_['cat']['encoder'].get_feature_names_out(categorical_columns)))

test_data_preprocessed_df = pd.DataFrame(test_data_preprocessed, columns=numerical_columns + binary_columns + ID_columns + list(preprocessor.named_transformers_['cat']['encoder'].get_feature_names_out(categorical_columns)))

# Model Building

## All Baseline Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=1),
    'Random Forest': RandomForestClassifier(random_state=1),
    #'Gradient Boosting': GradientBoostingClassifier(random_state=1),
    'LightGBM': LGBMClassifier(random_state=1),
    'XGBoost': XGBClassifier(random_state=1),
    'Decision Tree': DecisionTreeClassifier(random_state=1),
    'AdaBoost': AdaBoostClassifier(random_state=1),
    'Bagging': BaggingClassifier(random_state=1),
}

# Dictionary to store results
results = {}
train_accuracies = []
test_accuracies = []


# Loop through each model
for name, model in models.items():
    print(f"Training {name} with default parameters...\n")

    # Cross-validation
    cv_scores = cross_val_score(model, X_train_preprocessed, y_train, cv=5)
    print(f"{name} Cross-Validation Scores: {cv_scores}\n")

    model.fit(X_train_preprocessed, y_train)

    y_pred = model.predict(X_test_preprocessed)
    y_train_pred = model.predict(X_train_preprocessed)

    train_accuracy = accuracy_score(y_train, y_train_pred)
    test_accuracy = accuracy_score(y_test, y_pred)

    train_accuracies.append({"Model": name, "Accuracy": train_accuracy})
    test_accuracies.append({"Model": name, "Accuracy": test_accuracy})

    results[name] = {
        'Train Accuracy': train_accuracy,
        'Test Accuracy': test_accuracy,
        'CV Score Mean': np.mean(cv_scores),
    }

    print(f"{name} Metrics:")
    print(f"  Train Accuracy: {train_accuracy:.4f}")
    print(f"  Test Accuracy: {test_accuracy:.4f}")
    print(f"  Average CV Score: {np.mean(cv_scores):.4f}")

    print(f"Classification Report for {name}:\n")
    print(classification_report(y_test, y_pred))

    print("=" * 50)

## Model Performance Visualization

In [ ]:
train_df = pd.DataFrame(train_accuracies)
test_df = pd.DataFrame(test_accuracies)

models_list = train_df["Model"].tolist()
train_acc_values = train_df["Accuracy"].tolist()
test_acc_values = test_df["Accuracy"].tolist()

bar_width = 0.4
x = np.arange(len(models_list))

plt.figure(figsize=(10, 5))
plt.bar(x - bar_width/2, train_acc_values, bar_width, label="Train Accuracy", color='orange', alpha=0.7)
plt.bar(x + bar_width/2, test_acc_values, bar_width, label="Test Accuracy", color='blue', alpha=0.7)

plt.xlabel("Models")
plt.ylabel("Accuracy")
plt.title("Model Performance Comparison")
plt.xticks(x, models_list, rotation=45)
plt.legend()
plt.show()

3 Models I choose based on the above performance are:
+ i) XGBoost
+ ii) LightGBM
+ iii) RandomForest

# Top 3 Models

## Model 1 - XGBoost

In [ ]:
xgb = XGBClassifier(random_state=42)
xgb.fit(X_train_preprocessed, y_train)
xgb_pred = xgb.predict(X_test_preprocessed)

xgb_train_acc = accuracy_score(y_train, xgb.predict(X_train_preprocessed))
xgb_test_acc = accuracy_score(y_test, xgb_pred)
print(f"Train Accuracy: {xgb_train_acc}")
print("-------------------------------")
print(f"Test Accuracy: {xgb_test_acc}")
print("--------------------------------")
print(classification_report(y_test, xgb_pred))

### Visualizing Feature Importance

In [ ]:
importance = xgb.feature_importances_

feature_importance = pd.DataFrame({'Feature': X_train_preprocessed_df.columns, 'Importance': importance})
feature_importance = feature_importance.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(50, 50))
sns.barplot(x='Importance', y='Feature', data=feature_importance, palette='viridis')
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Feature Importance")
plt.show()

### Recursive Feature Elimination for XGBoost

In [ ]:
rfecv_xgb = RFECV(estimator=xgb, step=2, cv=5, scoring='accuracy')
rfecv_xgb.fit(X_train_preprocessed, y_train)

xgb_selected_features = X_train_preprocessed_df.columns[rfecv_xgb.support_]

X_train_preprocessed_xgb_selected = X_train_preprocessed_df[xgb_selected_features]
X_test_preprocessed_xgb_selected = X_test_preprocessed_df[xgb_selected_features]

xgb_selected = XGBClassifier(random_state=42)
xgb_selected.fit(X_train_preprocessed_xgb_selected, y_train)
xgb_selected_pred = xgb_selected.predict(X_test_preprocessed_xgb_selected)

train_acc = accuracy_score(y_train, xgb_selected.predict(X_train_preprocessed_xgb_selected))
test_acc = accuracy_score(y_test, xgb_selected_pred)

print(f"Train Accuracy: {train_acc}")
print("-------------------------------")
print(f"Test Accuracy: {test_acc}")
print("--------------------------------")
print(classification_report(y_test, xgb_selected_pred))


### XGBoost Hyperparameter Tuning

In [ ]:
xgb_param_grid = {
    'max_depth': [5, 7],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [100, 200],
    'reg_lambda': [10, 30]
}

xgb_grid_search = GridSearchCV(XGBClassifier(random_state=42), xgb_param_grid, scoring="accuracy", cv=5, n_jobs=-1, verbose=1)

xgb_grid_search.fit(X_train_preprocessed_xgb_selected, y_train)

print("Best Parameters:", xgb_grid_search.best_params_)

+ Best parameters XGBoost: 'max_depth'=5, 'learning_rate'=0.1, 'n_estimators'=200, 'reg_lambda'=10

In [ ]:
best_xgb = XGBClassifier(**xgb_grid_search.best_params_, random_state=42)
best_xgb.fit(X_train_preprocessed_xgb_selected, y_train)

xgb_train_acc_hy = accuracy_score(y_train, best_xgb.predict(X_train_preprocessed_xgb_selected))
xgb_test_acc_hy = accuracy_score(y_test, best_xgb.predict(X_test_preprocessed_xgb_selected))

print(f"Train Accuracy: {xgb_train_acc_hy}")
print(f"Test Accuracy: {xgb_test_acc_hy}")

## Model 2 - LightGBM

In [ ]:
lgbm = LGBMClassifier(random_state=42)
lgbm.fit(X_train_preprocessed, y_train)
lgbm_pred = lgbm.predict(X_test_preprocessed)

lgbm_train_acc = accuracy_score(y_train, lgbm.predict(X_train_preprocessed))
lgbm_test_acc = accuracy_score(y_test, lgbm_pred)

print(f"Train Accuracy: {lgbm_train_acc}")
print("-------------------------------")
print(f"Test Accuracy: {lgbm_test_acc}")
print("--------------------------------")
print(classification_report(y_test, lgbm_pred))

### Feature Importance

In [ ]:
importance = lgbm.feature_importances_

feature_importance = pd.DataFrame({'Feature': X_train_preprocessed_df.columns, 'Importance': importance})
feature_importance = feature_importance.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(50, 50))
sns.barplot(x='Importance', y='Feature', data=feature_importance, palette='viridis')
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Feature Importance")
plt.show()

### Recursive Feature Elimination

In [ ]:
rfecv_lgbm = RFECV(estimator=lgbm, step=2, cv=5, scoring='accuracy')
rfecv_lgbm.fit(X_train_preprocessed, y_train)

lgbm_selected_features = X_train_preprocessed_df.columns[rfecv_lgbm.support_]

X_train_preprocessed_lgbm_selected = X_train_preprocessed_df[lgbm_selected_features]
X_test_preprocessed_lgbm_selected = X_test_preprocessed_df[lgbm_selected_features]

lgbm_selected = LGBMClassifier(random_state=42)
lgbm_selected.fit(X_train_preprocessed_lgbm_selected, y_train)
lgbm_selected_pred = lgbm_selected.predict(X_test_preprocessed_lgbm_selected)

train_acc = accuracy_score(y_train, lgbm_selected.predict(X_train_preprocessed_lgbm_selected))
test_acc = accuracy_score(y_test, lgbm_selected_pred)

print(f"Train Accuracy: {train_acc}")
print("-------------------------------")
print(f"Test Accuracy: {test_acc}")
print("--------------------------------")
print(classification_report(y_test, lgbm_selected_pred))

### LightGBM Hyperparameter Tuning

In [ ]:
lgbm_param_grid = {
    'max_depth': [5, 7],
    'num_leaves': [15, 31],
    'learning_rate': [0.05, 0.1],
    'reg_lambda': [5, 10],
    'min_child_samples': [15, 30]
}

lgbm_grid_search = GridSearchCV(LGBMClassifier(random_state=42), lgbm_param_grid, scoring="accuracy", cv=5, n_jobs=-1, verbose=1)
lgbm_grid_search.fit(X_train_preprocessed_lgbm_selected, y_train)

print("Best Parameters:", lgbm_grid_search.best_params_)

+ Best Parameters LGBM: {'learning_rate': 0.1, 'max_depth': 7, 'min_child_samples': 30, 'num_leaves': 31, 'reg_lambda': 10}

In [ ]:
best_lgbm = LGBMClassifier(learning_rate=0.1, max_depth=7, min_child_samples=30, num_leaves=31, reg_lambda=10, random_state=42)
best_lgbm.fit(X_train_preprocessed_lgbm_selected, y_train)

lgbm_train_acc_hy = accuracy_score(y_train, best_lgbm.predict(X_train_preprocessed_lgbm_selected))
lgbm_test_acc_hy = accuracy_score(y_test, best_lgbm.predict(X_test_preprocessed_lgbm_selected))

print(f"Train Accuracy: {lgbm_train_acc_hy}")
print(f"Test Accuracy: {lgbm_test_acc_hy}")


In [ ]:
lgbm_selected = LGBMClassifier(random_state=42, learning_rate=0.1, reg_lambda=10, max_depth=7, min_child_samples=30, num_leaves=31)
lgbm_selected.fit(X_train_preprocessed_lgbm_selected, y_train)
lgbm_selected_pred = lgbm_selected.predict(X_test_preprocessed_lgbm_selected)

train_acc = accuracy_score(y_train, lgbm_selected.predict(X_train_preprocessed_lgbm_selected))
test_acc = accuracy_score(y_test, lgbm_selected_pred)

print(f"Train Accuracy: {train_acc}")
print("-------------------------------")
print(f"Test Accuracy: {test_acc}")
print("--------------------------------")
print(classification_report(y_test, lgbm_selected_pred))

## Model 3 - Random Forest

In [ ]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_preprocessed, y_train)
rf_pred = rf.predict(X_test_preprocessed)

rf_train_acc = accuracy_score(y_train, rf.predict(X_train_preprocessed))
rf_test_acc = accuracy_score(y_test, rf_pred)

print("Train acc", rf_train_acc)
print("Test acc", rf_test_acc)

### Recursive Feature Elimination

In [ ]:
rfecv_rf = RFECV(estimator=RandomForestClassifier(random_state=42), step=10, cv=2, scoring='accuracy')
rfecv_rf.fit(X_train_preprocessed, y_train)

selected_features_rf = X_train_preprocessed_df.columns[rfecv_rf.support_]

X_train_preprocessed_rf_selected = X_train_preprocessed_df[selected_features_rf]
X_test_preprocessed_rf_selected = X_test_preprocessed_df[selected_features_rf]

rf_selected = RandomForestClassifier(random_state=42)
rf_selected.fit(X_train_preprocessed_rf_selected, y_train)
rf_selected_pred = rf_selected.predict(X_test_preprocessed_rf_selected)

train_acc = accuracy_score(y_train, rf_selected.predict(X_train_preprocessed_rf_selected))
test_acc = accuracy_score(y_test, rf_selected_pred)

print(f"Train Accuracy: {train_acc}")
print("-------------------------------")
print(f"Test Accuracy: {test_acc}")
print("--------------------------------")
print(classification_report(y_test, rf_selected_pred))


### Random Forest Hyperparameter Tuning

In [ ]:
rf_param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [8, 10],
    'min_samples_leaf': [10, 20]
}

random_search = RandomizedSearchCV(RandomForestClassifier(random_state=42), rf_param_grid, n_iter=5, cv=2, random_state=42, n_jobs=-1)

random_search.fit(X_train_preprocessed_rf_selected, y_train)

print("Best Parameters:", random_search.best_params_)

+ Best parameters RandomForest: max_depth=8, n_estimators=50, min_samples_leaf=10

In [ ]:
best_rf = RandomForestClassifier(n_estimators=50, max_depth=8, min_samples_leaf=10, random_state=42)
best_rf.fit(X_train_preprocessed_rf_selected, y_train)

rf_train_acc_hy = accuracy_score(y_train, best_rf.predict(X_train_preprocessed_rf_selected))
rf_test_acc_hy = accuracy_score(y_test, best_rf.predict(X_test_preprocessed_rf_selected))

print(f"Train Accuracy: {rf_train_acc_hy}")
print(f"Test Accuracy: {rf_test_acc_hy}")

In [ ]:
before_acc = [xgb_test_acc, lgbm_test_acc, rf_test_acc]
hy_acc = [xgb_test_acc_hy, lgbm_test_acc_hy, rf_test_acc_hy]

models = ["XGBoost", "LightGBM", "Random Forest"]

x = np.arange(len(models))

width = 0.3

plt.figure(figsize=(8, 5))
plt.bar(x - width/2, before_acc, width, label='Before Tuning', color='blue')
plt.bar(x + width/2, hy_acc, width, label='After Tuning', color='green')


plt.xlabel('Models')
plt.ylabel('Accuracy')
plt.title('Model Accuracy Before vs After Hyperparameter Tuning')
plt.xticks(x, models)
plt.legend()

for i in range(len(models)):
    plt.text(x[i] - width/2, before_acc[i] + 0.005, f'{before_acc[i]:.4f}', ha='center', fontsize=10)
    plt.text(x[i] + width/2, hy_acc[i] + 0.005, f'{hy_acc[i]:.4f}', ha='center', fontsize=10)

plt.show()

# Prediction using best model

In [ ]:
lgbm_selected_features = X_train_preprocessed_df.columns[rfecv_lgbm.support_]

X_train_preprocessed_selected = X_train_preprocessed_df[lgbm_selected_features]
test_data_preprocessed_selected = test_data_preprocessed_df[lgbm_selected_features]

best_lgbm = LGBMClassifier(random_state=42, learning_rate=0.1, reg_lambda=10, max_depth=7, min_child_samples=30, num_leaves=31)
best_lgbm.fit(X_train_preprocessed_selected, y_train)
best_lgbm_pred = best_lgbm.predict(test_data_preprocessed_selected)

# Submission

In [ ]:
submission = pd.DataFrame(

    {"id": range(0, 10000),
     "target": best_lgbm_pred})

In [ ]:
submission.to_csv('submission.csv', index=False)

In [ ]:
submission.head()

In [ ]:
# import os

# os.remove('/kaggle/working/submission.csv')




#